# Includes

In [2]:
import pandas as pd
import numpy as np
import mne
import mne_nirs
import h5py
import shutil

from pathlib import Path
from mne.preprocessing.nirs import source_detector_distances, short_channels

root = Path.home() / "fnirs-representation-learning"
rs_data_dir = root / "snirf_dataset_2"

/home/asunkari/miniconda3/envs/neuro-ml/lib/python3.12/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
for subjdir in rs_data_dir.glob("Subj*"):
    if not subjdir.is_dir():
        continue

    rs_file_name = subjdir / "resting.snirf"
    clean_file = subjdir / "resting_clean.snirf"

    shutil.copy2(rs_file_name, clean_file)

    with h5py.File(clean_file, "r+") as f:
        del f["nirs"]["stim1"]

    print("created:", clean_file)

created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj96/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj98/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj97/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj95/resting_clean.snirf
created: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf
created: /home/asunkari/fn

In [29]:
raw_rest = mne.io.read_raw_snirf(subject_dir/"resting_clean.snirf", preload=True)

print(raw_rest.info["sfreq"])
print(raw_rest.times[-1])

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
Reading 0 ... 36799  =      0.000 ...   735.980 secs...
50.0
735.98


/tmp/ipykernel_317495/1220789148.py:1: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_rest = mne.io.read_raw_snirf(subject_dir/"resting_clean.snirf", preload=True)


In [30]:
# pick all fNIRS channels, then keep only CW amplitude channels
picks_fnirs = mne.pick_types(raw_rest.info, fnirs=True)
channel_types = np.array(raw_rest.get_channel_types())

picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]
print("fNIRS channels:", len(picks_fnirs))
print("CW amplitude channels:", len(picks_cw))

fNIRS channels: 112
CW amplitude channels: 112


In [31]:
# compute source-detector distances in meters
dists = source_detector_distances(raw_rest.info, picks=picks_cw)

# short channel mask using your thesis threshold
ss_mask_all = short_channels(raw_rest.info, threshold=0.015)
ss_mask = ss_mask_all[picks_cw]

# long channel mask using your thesis threshold
ls_mask = dists >= 0.025

print("distance range (m):", dists.min(), "to", dists.max())
print("CW channels total:", len(picks_cw))
print("SS CW channels:", int(ss_mask.sum()))
print("LS CW channels:", int(ls_mask.sum()))

distance range (m): 0.008 to 0.03046309242345563
CW channels total: 112
SS CW channels: 16
LS CW channels: 96


In [32]:
cw_names = np.array(raw_rest.ch_names)[picks_cw]
pair_names = np.array([name.split(" ")[0] for name in cw_names])

pair_table = pd.DataFrame({
    "channel_name": cw_names,
    "pair_name": pair_names,
    "distance_m": dists,
    "is_ss": ss_mask,
    "is_ls": ls_mask,
})

pair_summary = (
    pair_table.groupby("pair_name", as_index=False)
    .agg(
        distance_m=("distance_m", "first"),
        is_ss=("is_ss", "first"),
        is_ls=("is_ls", "first"),
    )
)

pair_summary["group"] = np.select(
    [pair_summary["is_ss"], pair_summary["is_ls"]],
    ["SS", "LS"],
    default="MID"
)

pair_summary["group"].value_counts()

group
LS    48
SS     8
Name: count, dtype: int64